[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/optimization/07_linear_quadratic_conic_programs/first_principles.ipynb)

# Topic 07: Linear, Quadratic & Conic Programs


## 1. First-Principles Intuition & Motivation

### 1.1 Why Structure Matters

General nonlinear optimization is hard: a smooth function can hide exponentially many local minima, and no algorithm can certify global optimality without extra structure.
The escape route is to *restrict the algebraic form* of the objective and constraints until global optimality becomes checkable.

Convexity is the first restriction: a convex objective over a convex feasible set has no spurious local minima.
But convexity alone is still too loose for industrial-strength solvers.
The classes studied here, **LP**, **QP**, **SOCP**, and **SDP**, restrict the form further, down to affine functions, positive semidefinite quadratics, norm cones, and matrix cones.
Each restriction buys something concrete: a duality theory with certificates, a geometric characterization of optima, and polynomial-time interior-point algorithms.


### 1.2 The Linear World

The oldest and most important structured class is the linear program.
Imagine allocating scarce resources: each decision variable $x_j \ge 0$ is a production level, each constraint row of $Ax \le b$ is a resource budget, and the cost $c^T x$ is linear because doubling production doubles cost.

Two first-principles observations drive the whole theory:

1. The feasible set $\{x : Ax \le b\}$ is an intersection of half-spaces, a **polyhedron**: a flat-sided convex body with finitely many corners.
2. A nonconstant affine objective has a constant, nonzero gradient: it always pays to keep sliding in the direction $-c$ until the boundary stops you.

Together these force an optimum (when one exists) to sit on a face of the polyhedron, and in particular at a **vertex**.
This single geometric fact explains why the simplex method, which only ever visits vertices, can solve LPs at all.


### 1.3 Climbing the Hierarchy

What changes when the cost curves upward?
Adding a positive semidefinite quadratic term $\tfrac{1}{2}x^T Q x$ produces the **QP** class, which captures least squares, ridge regression, support vector machines, and Markowitz portfolios.
Allowing constraints of the form "a norm of an affine expression is at most an affine expression" produces the **SOCP** class, which captures robust linear programming under ellipsoidal uncertainty.
Finally, replacing scalar inequalities by the requirement that an affine *matrix-valued* function be positive semidefinite produces the **SDP** class, the most expressive tractable cone class known, powering control theory (LMIs) and combinatorial relaxations such as max-cut.

The punchline of this module is that these are not four unrelated formats but one nested chain,

$$
\text{LP} \subset \text{QP} \subset \text{SOCP} \subset \text{SDP},
$$

and every inclusion is witnessed by an explicit, mechanical reformulation that we will prove in Section 3.
Knowing the chain tells you the cheapest solver class that can express your model.


## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 1 (Linear program: inequality and standard forms).**
A *linear program* in **inequality form** is

$$
\min_{x \in \mathbb{R}^n} \ c^T x \quad \text{subject to} \quad Ax \le b,
$$

with $c \in \mathbb{R}^n$, $A \in \mathbb{R}^{m \times n}$, $b \in \mathbb{R}^m$.
A linear program in **standard form** is

$$
\min_{x \in \mathbb{R}^n} \ c^T x \quad \text{subject to} \quad Ax = b, \quad x \ge 0.
$$

The two forms are interconvertible by two mechanical constructions:

- **Slack variables**: each inequality $a_i^T x \le b_i$ becomes the pair $a_i^T x + s_i = b_i$, $s_i \ge 0$; the new variable $s_i$ measures the unused budget of constraint $i$.
- **Free-variable splitting**: an unrestricted variable $x_j$ is replaced by $x_j = x_j^+ - x_j^-$ with $x_j^+ \ge 0$ and $x_j^- \ge 0$.

Both maps preserve feasibility and objective values, so the optimal value is unchanged; they only re-parameterize the geometry.


**Definition 2 (Polyhedron, extreme point, basic feasible solution).**
A *polyhedron* is any set of the form $P = \{x \in \mathbb{R}^n : Ax \le b\}$; it is convex because it is an intersection of half-spaces.

A point $v \in P$ is an *extreme point* (vertex) if it cannot be written as $v = \tfrac{1}{2}(y + z)$ with $y, z \in P$ and $y \neq z$.

For the standard-form feasible set $\mathcal{F} = \{x : Ax = b,\ x \ge 0\}$ with $A \in \mathbb{R}^{m \times n}$ of full row rank, a *basic feasible solution* (BFS) is a feasible $x$ whose support $S = \{j : x_j \gt 0\}$ indexes linearly independent columns $\{A_j : j \in S\}$.
Equivalently, one selects a basis $B$ of $m$ independent columns, sets the nonbasic variables to zero, and solves $A_B x_B = b$; the BFS is feasible when $x_B \ge 0$.

**Fact.** For standard-form polyhedra, extreme points and basic feasible solutions coincide (proved in the exercises), and a polyhedron has finitely many of them: at most $\binom{n}{m}$.


**Theorem 1 (Fundamental theorem of linear programming).**
Consider a standard-form LP over the nonempty feasible set $\mathcal{F} = \{x : Ax = b,\ x \ge 0\}$.

1. If the LP has a finite optimal value attained at some feasible point, then it is attained at some *extreme point* of $\mathcal{F}$.
2. Consequently an optimal solution can be found by examining only the finitely many basic feasible solutions.

This theorem converts a continuous search over $\mathbb{R}^n$ into a combinatorial search over vertices, and it is the mathematical license behind the simplex method.
The proof (Section 3.1) works by taking an optimal point of minimal support and showing that any linear dependence among its support columns can be exploited to shrink the support without losing optimality.


**Theorem 2 (LP duality and complementary slackness).**
To every primal LP corresponds a dual LP built on the same data.
The symmetric-form correspondence table is:

| Item | Primal | Dual |
|---|---|---|
| Objective | minimize $c^T x$ | maximize $b^T y$ |
| Constraints | $Ax \ge b$ | $A^T y \le c$ |
| Sign restriction | $x \ge 0$ | $y \ge 0$ |
| Data roles | costs $c$, resources $b$ | prices $y$ on resources |

For the standard form $\min\{c^T x : Ax = b,\ x \ge 0\}$ the dual is $\max\{b^T y : A^T y \le c\}$ with $y$ free.

- **Weak duality**: every primal-feasible $x$ and dual-feasible $y$ satisfy $b^T y \le c^T x$.
- **Strong duality**: if either problem has a finite optimum, both do, and the optimal values are equal.
- **Complementary slackness**: feasible $x^*$ and $y^*$ are simultaneously optimal if and only if $x_j^* (c - A^T y^*)_j = 0$ for every $j$ (and, in inequality form, $y_i^* (Ax^* - b)_i = 0$ for every $i$).

The dual variables $y_i^*$ are *shadow prices*: under nondegeneracy, $y_i^*$ is the rate of change of the optimal cost per unit change in $b_i$.


**Definition 3 (Quadratic program).**
A *quadratic program* is

$$
\min_{x \in \mathbb{R}^n} \ \tfrac{1}{2} x^T Q x + c^T x \quad \text{subject to} \quad Ax \le b, \quad Ex = d,
$$

where $Q = Q^T \succeq 0$.
Positive semidefiniteness of $Q$ is exactly the condition making the objective convex, hence making the QP globally solvable; an indefinite $Q$ yields an NP-hard problem.

**Least squares is the canonical unconstrained QP.**
Expanding the residual norm of a linear model $Xw \approx y$,

$$
\tfrac{1}{2}\lVert Xw - y \rVert_2^2 = \tfrac{1}{2} w^T (X^T X) w - (X^T y)^T w + \tfrac{1}{2} y^T y,
$$

identifies $Q = X^T X \succeq 0$ and $c = -X^T y$.
Section 3.3 shows that the minimizer solves the *normal equations* $(X^T X) w = X^T y$.

**Markowitz portfolio selection** is the canonical constrained QP: minimize portfolio variance $w^T \Sigma w$ subject to a target return $\mu^T w \ge R$ and the budget $\mathbf{1}^T w = 1$, where $\Sigma \succeq 0$ is the return covariance matrix.


**Definition 4 (Second-order cone and SOCP).**
The *second-order cone* (Lorentz cone, ice-cream cone) in $\mathbb{R}^{k+1}$ is

$$
\mathcal{Q}^{k+1} = \{(u, t) \in \mathbb{R}^k \times \mathbb{R} : \lVert u \rVert_2 \le t\}.
$$

A *second-order cone program* is

$$
\min_{x} \ f^T x \quad \text{subject to} \quad \lVert A_i x + b_i \rVert_2 \le c_i^T x + d_i, \quad i = 1, \dots, m,
$$

that is, each constraint requires the affine image $(A_i x + b_i,\ c_i^T x + d_i)$ to lie in a second-order cone.
Taking $A_i = 0$ recovers ordinary linear inequalities, so every LP is an SOCP.

**Robust LP as SOCP.**
Suppose a constraint row $a^T x \le b$ has an uncertain coefficient vector $a \in \mathcal{E} = \{\bar{a} + P u : \lVert u \rVert_2 \le 1\}$ (an ellipsoid), and we demand feasibility for *every* realization.
The worst case is

$$
\max_{\lVert u \rVert_2 \le 1} (\bar{a} + P u)^T x = \bar{a}^T x + \lVert P^T x \rVert_2,
$$

so the robust constraint is exactly the second-order cone constraint $\bar{a}^T x + \lVert P^T x \rVert_2 \le b$.
Immunizing an LP against ellipsoidal data uncertainty therefore promotes it one level up the hierarchy, to an SOCP.


**Definition 5 (Semidefinite program and LMI).**
Let $\mathbb{S}^n$ denote symmetric $n \times n$ matrices and $X \succeq 0$ mean $X$ is positive semidefinite.
A *semidefinite program* in standard primal form is

$$
\min_{X \in \mathbb{S}^n} \ \operatorname{tr}(C X) \quad \text{subject to} \quad \operatorname{tr}(A_i X) = b_i \ (i = 1, \dots, m), \quad X \succeq 0.
$$

The inequality-form counterpart uses a *linear matrix inequality* (LMI): $F(x) = F_0 + x_1 F_1 + \dots + x_n F_n \succeq 0$ with symmetric data matrices $F_j$.
The feasible set of an LMI is convex because the PSD cone is convex and $F$ is affine.

**Max-cut relaxation (mention).**
For a graph with weights $w_{ij}$, the max-cut problem maximizes $\tfrac{1}{2}\sum_{i \lt j} w_{ij}(1 - x_i x_j)$ over labels $x_i \in \{-1, +1\}$.
Writing $X = x x^T$ and dropping the rank-one requirement leaves the SDP relaxation: maximize $\tfrac{1}{2}\sum_{i \lt j} w_{ij}(1 - X_{ij})$ subject to $X_{ii} = 1$ and $X \succeq 0$, the basis of the Goemans-Williamson $0.878$ approximation guarantee.

**Theorem 3 (Convex hierarchy).**
Every LP is a QP, every convex QP can be reformulated as an SOCP, and every SOCP can be reformulated as an SDP:

$$
\text{LP} \subset \text{QP} \subset \text{SOCP} \subset \text{SDP}.
$$

Each inclusion is constructive (Section 3.6) and strict: each class contains problems not expressible in the previous one.


## 3. Step-by-Step Mathematical Proofs & Derivations

### 3.1 Proof: Fundamental Theorem of LP (an optimal extreme point exists)

**Claim.** If the standard-form LP $\min\{c^T x : Ax = b,\ x \ge 0\}$ attains its optimum, then some extreme point of the feasible set is optimal.

**Proof.**
Among all optimal solutions, choose $x^*$ with the *smallest number of positive components*, and let $S = \{j : x_j^* \gt 0\}$ be its support.
We show the columns $\{A_j : j \in S\}$ are linearly independent, which makes $x^*$ a basic feasible solution, hence an extreme point.

Suppose instead the columns are dependent: there exists $d \neq 0$ supported on $S$ with $Ad = 0$.
For small $\lvert\theta\rvert$ the point $x(\theta) = x^* + \theta d$ satisfies $A x(\theta) = b$ and stays nonnegative, because every component of $x^*$ on $S$ is strictly positive.

*Step 1: the direction is cost-neutral.*
The objective along the segment is $c^T x(\theta) = c^T x^* + \theta\, c^T d$.
If $c^T d \neq 0$, then choosing $\theta$ small with sign opposite to $c^T d$ produces a feasible point with strictly smaller cost, contradicting optimality of $x^*$.
Hence $c^T d = 0$: every point $x(\theta)$ that remains feasible is also optimal.

*Step 2: slide until a component dies.*
Replacing $d$ by $-d$ if necessary, assume $d$ has at least one negative component ($d \neq 0$ and, if $d \ge 0$, use $-d$; if both $d \ge 0$ and $-d \ge 0$ then $d = 0$, a contradiction).
Define

$$
\theta^* = \min_{j : d_j \lt 0} \frac{x_j^*}{-d_j} \gt 0.
$$

At $\theta = \theta^*$ the point $x(\theta^*)$ is feasible, still optimal by Step 1, and has at least one more zero component than $x^*$ (the minimizing index $j$ drops out of the support), contradicting the minimality of the support of $x^*$.

Therefore no such $d$ exists, the support columns are independent, and $x^*$ is a basic feasible solution, i.e. an extreme point of the polyhedron.

$$
\boxed{\text{If an LP attains its optimum, some extreme point (vertex) of the feasible polyhedron is optimal.}}
$$


### 3.2 Proof: LP Weak Duality and Complementary Slackness

**Setting.** Primal: $\min\ c^T x$ subject to $Ax = b$, $x \ge 0$. Dual: $\max\ b^T y$ subject to $A^T y \le c$.

**Step 1 (weak duality).**
Let $x$ be primal feasible and $y$ dual feasible.
Then

$$
c^T x - b^T y = c^T x - (Ax)^T y = x^T c - x^T A^T y = x^T (c - A^T y).
$$

Both factors are componentwise nonnegative: $x \ge 0$ by primal feasibility and $c - A^T y \ge 0$ by dual feasibility.
A sum of products of nonnegative numbers is nonnegative, hence

$$
c^T x - b^T y = \sum_{j=1}^{n} x_j \,(c - A^T y)_j \ \ge\ 0 .
$$

So every dual-feasible $y$ certifies the lower bound $b^T y \le c^T x$ for every primal-feasible $x$; in particular $d^* \le p^*$.

**Step 2 (complementary slackness).**
Suppose $x^*$ and $y^*$ are feasible and achieve equality $c^T x^* = b^T y^*$.
Then the sum above vanishes:

$$
\sum_{j=1}^{n} x_j^* \,(c - A^T y^*)_j = 0,
$$

and since every term is nonnegative, *each term is zero*: for all $j$, either $x_j^* = 0$ or $(c - A^T y^*)_j = 0$.
Conversely, if each term vanishes, the objectives are equal, and by weak duality no feasible pair can do better, so both points are optimal.

**Step 3 (interpretation).**
A positive primal variable forces its dual constraint to be tight (the activity is priced exactly), and a slack dual constraint forces its primal variable to zero (an unprofitable activity is unused).

$$
\boxed{\,b^T y \le c^T x \ \text{ for all feasible pairs, with equality} \iff x_j^*\,(c - A^T y^*)_j = 0 \ \forall j.\,}
$$


### 3.3 Proof: Unconstrained QP, Completing the Square, and the Normal Equations

**Claim.** For $f(x) = \tfrac{1}{2} x^T Q x + c^T x$ with $Q \succ 0$, the unique global minimizer is $x^* = -Q^{-1} c$, and applying this to least squares yields the normal equations.

**Step 1 (complete the square).**
Since $Q \succ 0$, the inverse $Q^{-1}$ exists. Write

$$
f(x) = \tfrac{1}{2}\,(x + Q^{-1} c)^T Q\, (x + Q^{-1} c) - \tfrac{1}{2}\, c^T Q^{-1} c .
$$

*Verification*: expanding the first term gives $\tfrac{1}{2} x^T Q x + x^T Q Q^{-1} c + \tfrac{1}{2} c^T Q^{-1} Q Q^{-1} c = \tfrac{1}{2} x^T Q x + c^T x + \tfrac{1}{2} c^T Q^{-1} c$, and subtracting $\tfrac{1}{2} c^T Q^{-1} c$ recovers $f(x)$ exactly.

**Step 2 (read off the minimizer).**
The second term is a constant. The first term is a $Q$-weighted squared norm: it is $\ge 0$ for every $x$ because $Q \succ 0$, and it equals $0$ *only* at $x + Q^{-1}c = 0$.
Hence the unique minimizer and minimal value are

$$
x^* = -Q^{-1} c, \qquad f(x^*) = -\tfrac{1}{2}\, c^T Q^{-1} c .
$$

Uniqueness is strict: any $x \neq x^*$ pays a positive quadratic penalty $\tfrac{1}{2}(x - x^*)^T Q (x - x^*) \gt 0$.

**Step 3 (application to least squares).**
For $f(w) = \tfrac{1}{2}\lVert X w - y \rVert_2^2$ we found $Q = X^T X$ and $c = -X^T y$.
If $X$ has full column rank then $Q \succ 0$, and Step 2 gives $w^* = (X^T X)^{-1} X^T y$; equivalently, the stationarity condition $Q w + c = 0$ reads

$$
\boxed{\,(X^T X)\, w = X^T y \quad \text{(normal equations)}, \qquad x^* = -Q^{-1} c \ \text{ for } Q \succ 0.\,}
$$


### 3.4 Proof: KKT Linear System of the Equality-Constrained QP

**Claim.** The problem $\min_x\ \tfrac{1}{2} x^T Q x + c^T x$ subject to $Ax = b$ (with $Q \succeq 0$, $A$ full row rank) is solved by a single symmetric *block linear system*.

**Step 1 (Lagrangian stationarity).**
Form the Lagrangian with multiplier $\lambda \in \mathbb{R}^m$:

$$
\mathcal{L}(x, \lambda) = \tfrac{1}{2} x^T Q x + c^T x + \lambda^T (A x - b).
$$

A constrained minimizer must be stationary in $x$:

$$
\nabla_x \mathcal{L} = Q x + c + A^T \lambda = 0 .
$$

**Step 2 (assemble with feasibility).**
Stationarity and primal feasibility $Ax = b$ are two affine equations in the unknowns $(x, \lambda)$; stacking them gives the *KKT system*

$$
\begin{bmatrix} Q & A^T \\ A & 0 \end{bmatrix}
\begin{bmatrix} x \\ \lambda \end{bmatrix}
=
\begin{bmatrix} -c \\ b \end{bmatrix}.
$$

**Step 3 (sufficiency).**
Let $(x^*, \lambda^*)$ solve the system and let $x$ be any feasible point, so $A(x - x^*) = 0$.
Convexity of the objective ($Q \succeq 0$) gives, with $g = Qx^* + c = -A^T \lambda^*$,

$$
f(x) - f(x^*) = g^T (x - x^*) + \tfrac{1}{2}(x - x^*)^T Q (x - x^*) = -\lambda^{*T} A (x - x^*) + \tfrac{1}{2}(x - x^*)^T Q (x - x^*) \ \ge\ 0,
$$

since the first term vanishes on the feasible subspace and the second is nonnegative.
Hence $x^*$ is a global constrained minimizer.

**Step 4 (nonsingularity).**
If moreover $Q \succ 0$ on the null space of $A$ (i.e. $z^T Q z \gt 0$ whenever $Az = 0$, $z \neq 0$) and $A$ has full row rank, the KKT matrix is nonsingular, so the solution $(x^*, \lambda^*)$ is unique.

$$
\boxed{\;\begin{bmatrix} Q & A^T \\ A & 0 \end{bmatrix}\begin{bmatrix} x^* \\ \lambda^* \end{bmatrix} = \begin{bmatrix} -c \\ b \end{bmatrix}\;}
$$


### 3.5 Proof: Schur Complement Lemma for PSD Block Matrices

**Claim.** Let $M = \begin{bmatrix} A & B \\ B^T & C \end{bmatrix}$ be symmetric with $A \succ 0$. Then

$$
M \succeq 0 \iff S = C - B^T A^{-1} B \succeq 0,
$$

where $S$ is the *Schur complement* of $A$ in $M$.

**Step 1 (block congruence).**
Define the invertible block-triangular matrix

$$
T = \begin{bmatrix} I & 0 \\ -B^T A^{-1} & I \end{bmatrix}.
$$

Direct multiplication gives the block diagonalization

$$
T M T^T = \begin{bmatrix} A & 0 \\ 0 & C - B^T A^{-1} B \end{bmatrix}.
$$

*Verification of the (2,2) block*: it equals $B^T A^{-1} A A^{-1} B - B^T A^{-1} B - B^T A^{-1} B + C = C - B^T A^{-1} B$, since the four block products contribute one $+B^T A^{-1}B$ term and two $-B^T A^{-1}B$ terms alongside $C$.

**Step 2 (congruence preserves semidefiniteness).**
For any invertible $T$, $M \succeq 0$ if and only if $T M T^T \succeq 0$: given $v$, set $w = T^T v$; then $v^T (T M T^T) v = w^T M w$, and as $v$ ranges over all vectors so does $w$.

**Step 3 (conclude).**
A block-diagonal symmetric matrix is PSD if and only if each diagonal block is PSD.
Since $A \succ 0$ already, $T M T^T \succeq 0$ reduces exactly to $S \succeq 0$.

$$
\boxed{\,A \succ 0 \implies \big( M \succeq 0 \iff C - B^T A^{-1} B \succeq 0 \big).\,}
$$

This lemma is the engine that converts quadratic and norm constraints into linear matrix inequalities in the next proof.


### 3.6 Proof: The Embedding Chain LP to QP to SOCP to SDP

**Step 1 (LP is a QP).**
Set $Q = 0$ in the QP objective $\tfrac{1}{2}x^T Q x + c^T x$: the zero matrix is symmetric PSD, and the objective degenerates to the linear cost $c^T x$ with the same linear constraints. Hence every LP is a QP.

**Step 2 (convex QP is an SOCP).**
Given $\min\ \tfrac{1}{2} x^T Q x + c^T x$ with $Q \succeq 0$, factor $Q = R^T R$ (Cholesky or eigen square root).
Introduce an epigraph scalar $s$ and rewrite the problem as

$$
\min_{x,\,s} \ \tfrac{1}{2} s + c^T x \quad \text{subject to} \quad \lVert R x \rVert_2^2 \le s .
$$

The constraint $\lVert u \rVert_2^2 \le s$ (with $s \ge 0$ implied) is turned into a second-order cone constraint by the *norm rotation identity*: for any vector $u$ and scalars $a, b \ge 0$,

$$
\lVert u \rVert_2^2 \le a\,b \iff \left\lVert \begin{bmatrix} 2u \\ a - b \end{bmatrix} \right\rVert_2 \le a + b ,
$$

which follows by squaring both sides: $4\lVert u\rVert_2^2 + (a-b)^2 \le (a+b)^2 \iff 4\lVert u\rVert_2^2 \le 4ab$.
Apply it with $u = Rx$, $a = s$, $b = 1$: the QP epigraph constraint becomes the cone constraint with affine entries $(2Rx,\ s - 1,\ s + 1)$. All original linear constraints stay linear, so the whole problem is an SOCP.

**Step 3 (SOCP is an SDP).**
It suffices to write one cone constraint $\lVert u \rVert_2 \le t$ (with $u = Ax + b$, $t = c^T x + d$ affine) as an LMI. Consider the *arrow matrix*

$$
M(u, t) = \begin{bmatrix} t I & u \\ u^T & t \end{bmatrix}.
$$

If $t \gt 0$, the Schur complement lemma (Section 3.5, applied to the block $tI \succ 0$) gives $M \succeq 0 \iff t - u^T (tI)^{-1} u \ge 0 \iff t^2 \ge \lVert u \rVert_2^2 \iff \lVert u \rVert_2 \le t$.
If $t = 0$, $M \succeq 0$ forces $u = 0$ (test vectors $(w, \pm 1)$), matching $\lVert u \rVert_2 \le 0$.
If $t \lt 0$, both conditions fail.
Since $M(u, t)$ is affine in $x$, the constraint is an LMI, and the SOCP becomes an SDP.

$$
\boxed{\text{LP} \subset \text{QP} \subset \text{SOCP} \subset \text{SDP}, \ \text{each inclusion witnessed by an explicit affine embedding.}}
$$


## 4. Computational & Algorithmic Insights

### 4.1 The Simplex Method: A Combinatorial Edge Walk

The fundamental theorem licenses searching only vertices. The simplex method (Dantzig, 1947) organizes that search:

1. Start at a basic feasible solution (found via an auxiliary "phase one" LP).
2. Compute *reduced costs* of nonbasic variables; if all are nonnegative, the current vertex is optimal (this is exactly a complementary-slackness certificate).
3. Otherwise, bring a negative-reduced-cost variable into the basis and slide along the corresponding edge until the *ratio test* forces a basic variable to zero; pivot and repeat.

Each pivot costs $O(mn)$ arithmetic on the tableau (or a rank-one factorization update).
Empirically the method finishes in $O(m)$ to $O(m + n)$ pivots on real instances, but the **Klee-Minty cube** shows a worst case of $2^n$ pivots: a deformed hypercube whose vertices the classical pivot rule visits exhaustively.
Smoothed analysis (Spielman-Teng) explains the gap: tiny random perturbations of any instance make the expected pivot count polynomial.


### 4.2 Interior-Point and Barrier Methods: The Analytic Route

Interior-point methods ignore vertices entirely. Replace $x \ge 0$ by a *logarithmic barrier* added to the objective:

$$
\min_x \ t\, c^T x - \sum_{j=1}^{n} \ln x_j \quad \text{subject to} \quad Ax = b .
$$

For each barrier weight $t \gt 0$ the minimizer $x^*(t)$ is strictly interior; the curve $\{x^*(t) : t \gt 0\}$ is the **central path**, and $x^*(t) \to x^*$ as $t \to \infty$ with duality gap exactly $n/t$ for LP.
A path-following method alternates: increase $t$ by a fixed factor, then re-center with a few Newton steps on the barrier problem.

- Self-concordance theory (Nesterov-Nemirovski) shows $O(\sqrt{n}\,\ln(1/\epsilon))$ Newton iterations suffice to reach accuracy $\epsilon$, each iteration solving one KKT-like linear system, giving overall polynomial complexity (about $O(n^{3.5} L)$ bit operations for LP in classical accounting).
- The *same template* solves QP, SOCP, and SDP by swapping the barrier: $-\ln x_j$ for the orthant, $-\ln(t^2 - \lVert u \rVert_2^2)$ for the second-order cone, $-\ln \det X$ for the PSD cone.

This is the computational payoff of the hierarchy: one Newton-based engine covers all four classes, with per-iteration cost growing from LP (cheapest) to SDP (most expensive, since the variable is a matrix).


### 4.3 Modeling Practice: Getting Problems into Conic Form

Practical pipeline for using these classes:

- **Recognize the class**: check whether the objective is affine or PSD-quadratic and whether constraints are affine, norm-affine, or LMI. Aim for the *lowest* class that expresses the model; solver speed and reliability degrade up the chain.
- **Standardize mechanically**: slacks for inequalities, splitting for free variables, epigraph variables for objectives inside constraints, Schur complements for quadratic-over-linear terms.
- **Mind conditioning**: normal-equation matrices $X^T X$ square the condition number of $X$; prefer QR or regularization for ill-conditioned least squares embedded in larger programs.
- **Exploit sparsity and structure**: simplex exploits basis sparsity; interior-point methods exploit sparse Cholesky of the KKT system; SDP solvers exploit chordal sparsity of the matrix variable.
- **Certificates**: always extract dual variables. They certify optimality (complementary slackness), price resources, and diagnose infeasibility via Farkas-type alternatives.

Rough problem-size guidance with modern solvers: LPs with millions of variables, QPs with hundreds of thousands, SOCPs with tens of thousands of cones, and SDPs with matrix blocks up to a few thousand are routinely solvable.


## 5. Real-World Physics & AI/ML Applications

### 5.1 Finance and Operations: Markowitz and Resource Allocation

The Markowitz mean-variance problem

$$
\min_{w} \ w^T \Sigma w \quad \text{subject to} \quad \mu^T w \ge R, \quad \mathbf{1}^T w = 1
$$

is a QP whose KKT system (Section 3.4) yields closed-form efficient frontiers; its dual multipliers price the return target ($\partial(\text{variance})/\partial R$) and reveal the trade-off slope of the frontier.
Classical operations research runs on LP: production planning, blending, transportation, and network flow, where dual prices drive real decisions (which machine hour is worth buying, which nutrient constraint drives the diet cost).
These models are solved daily at extreme scale by simplex and barrier codes.


### 5.2 Machine Learning: Least Squares, Ridge, and SVM

- **Linear regression** is the unconstrained QP of Section 3.3: $(X^T X) w = X^T y$.
- **Ridge regression** adds $\tfrac{\lambda}{2}\lVert w \rVert_2^2$, shifting the Hessian to $X^T X + \lambda I \succ 0$: always strictly convex, always uniquely solvable, better conditioned.
- **Hard-margin SVM** maximizes the margin $2 / \lVert w \rVert_2$ between classes, equivalently

$$
\min_{w, b} \ \tfrac{1}{2}\lVert w \rVert_2^2 \quad \text{subject to} \quad y_i (w^T x_i + b) \ge 1 \ \ \forall i,
$$

a QP with PSD Hessian $\operatorname{diag}(I, 0)$; the soft-margin version adds slack variables (an LP-style construction) and remains a QP. Its dual QP over multipliers $\alpha_i$ exposes support vectors through complementary slackness: only points with tight margin constraints get $\alpha_i \gt 0$.
- **LASSO and sparse learning** reformulate the $\ell_1$ penalty with variable splitting into a QP; robust regression variants become SOCPs.


### 5.3 Engineering, Control, and Combinatorics: SOCP and SDP at Work

- **Robust engineering design**: tolerancing and antenna/filter design under ellipsoidal parameter uncertainty are SOCPs via the robust-LP construction of Definition 4; truss topology design is a classic SDP/SOCP family.
- **Control theory**: Lyapunov stability of $\dot{z} = A z$ is the LMI feasibility problem $A^T P + P A \prec 0$, $P \succ 0$; $H_\infty$ synthesis and many observer designs are SDPs. This is the physical face of the PSD cone: energy functions decreasing along trajectories.
- **Combinatorial relaxation**: the max-cut SDP (Definition 5) relaxes $x_i \in \{-1, 1\}$ to unit vectors; random-hyperplane rounding of the SDP solution guarantees at least $0.878$ of the optimum cut (Goemans-Williamson), the flagship example of SDP bounding an NP-hard problem.
- **Quantum and statistical physics**: density matrices are PSD with unit trace, so state estimation and energy minimization over quantum states are SDPs; moment relaxations of polynomial optimization (Lasserre hierarchy) climb further up the same conic ladder.


## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Where |
|---|---|---|
| LP, QP, SOCP, SDP standard forms and conversions | Boyd & Vandenberghe, *Convex Optimization* | Chapter 4; Appendix A.5.5 (Schur complements) |
| Simplex method, degeneracy, LP duality | Dantzig, *Linear Programming and Extensions* | Chapters 5-7 |
| Simplex and interior-point implementations; QP and KKT systems | Nocedal & Wright, *Numerical Optimization* (2nd ed.) | Chapters 13-14, 16 |
| Basic feasible solutions, fundamental theorem, barrier methods | Luenberger & Ye, *Linear and Nonlinear Programming* (4th ed.) | Chapters 2-5 |
| Conic duality, expressiveness of SOCP/SDP, self-concordance | Ben-Tal & Nemirovski, *Lectures on Modern Convex Optimization* | Lectures 2-4 |
| SDP theory, LMIs, applications, max-cut relaxation | Vandenberghe & Boyd, "Semidefinite Programming," *SIAM Review* 38(1) | Whole survey |
| Mean-variance portfolio QP | Markowitz, "Portfolio Selection," *J. Finance* 7(1), 1952 | Whole paper |

Suggested reading order: Boyd & Vandenberghe Chapter 4 for the taxonomy, Luenberger & Ye for polyhedral geometry and the fundamental theorem, Nocedal & Wright for algorithms, then Ben-Tal & Nemirovski and the SDP survey for the conic frontier.
